# Pashto XLM-R: shared splits and validation-based checkpoint selection
Run these cells in order in Google Colab. Select **Runtime → Change runtime type → GPU** first.
Upload the existing `train_data.csv` and `test_data.csv` to the same Google Drive folder.

Important steps:
1. Load saved CSVs.
2. Hold out 10% of the training CSV for validation.
3. Select the best of four epoch checkpoints using validation macro-F1.
4. Predict the shared test set only after training and checkpoint selection.
5. Save article IDs, predictions, reports, configuration, tokenizer, and the selected model.


## Step 1 — Install dependencies
Uses a fixed Transformers version for the APIs below. If Colab asks to restart the runtime, restart and continue from Step 2.

In [ ]:
%pip install -q "transformers==4.57.1" "datasets>=3,<5" "accelerate>=1.1,<2" sentencepiece scikit-learn pandas matplotlib seaborn


## Step 2 — Mount Drive and configure paths (updated)
Edit `DATA_DIR` to the folder containing your two shared CSVs. Outputs are saved in a new timestamped Drive folder.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone

drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/Pashto_Dataset')  # EDIT THIS
TRAIN_PATH = DATA_DIR / 'train_data.csv'
TEST_PATH = DATA_DIR / 'test_data.csv'
RUN_NAME = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
OUTPUT_DIR = DATA_DIR / f'xlmr_results_{RUN_NAME}'
for path in (TRAIN_PATH, TEST_PATH):
    if not path.is_file():
        raise FileNotFoundError(f'File not found: {path}. Upload your shared CSVs and edit DATA_DIR.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print('Results folder:', OUTPUT_DIR)


## Step 3 — Import packages and set the seed
The seed is set before model initialization. GPU execution may still vary across runs.

In [ ]:
import hashlib
import json
import platform
import time
import numpy as np
import pandas as pd
import torch
import transformers
import datasets
import sklearn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, set_seed

SEED = 42
MODEL_NAME = 'xlm-roberta-base'
MAX_LENGTH = 256
set_seed(SEED)
print('Transformers:', transformers.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available():
    print('For practical training time, enable a Colab GPU before proceeding.')


## Step 4 — Load shared partitions
Text is preserved. Existing `article_id` is used, with `body_hash` as a fallback. The ID overlap check validates partition separation; it does not perform deduplication.

In [ ]:
def load_partition(path):
    df = pd.read_csv(path, encoding='utf-8-sig', keep_default_na=False)
    if df.empty:
        raise ValueError(f'Empty partition: {path}')
    for field in ('text', 'category'):
        if field not in df or not df[field].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
            raise ValueError(f'{path}: {field} must contain nonempty strings')
    if 'article_id' not in df:
        if 'body_hash' not in df:
            raise ValueError(f'{path}: expected article_id or body_hash')
        df['article_id'] = df['body_hash']
    if not df['article_id'].map(lambda x: isinstance(x, str) and bool(x.strip())).all():
        raise ValueError(f'{path}: invalid article_id')
    return df

train_df = load_partition(TRAIN_PATH)
test_df = load_partition(TEST_PATH)
if set(train_df['article_id']) & set(test_df['article_id']):
    raise ValueError('Shared training and test partitions overlap by article_id')
label_list = sorted(train_df['category'].unique())
if len(label_list) < 3 or not set(test_df['category']).issubset(label_list):
    raise ValueError('Expected at least three training classes and known test labels')
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}
print('Shared training articles:', len(train_df))
print('Shared test articles:', len(test_df))
print('Labels:', label_list)
print('Training category counts:')
print(train_df['category'].value_counts())


## Step 5 — Create validation data within training
10% of the shared training partition is used for checkpoint selection. The shared test partition stays unchanged. With 19,297 training articles, this gives 17,367 for weight updates and 1,930 for validation. These subsets are saved for reproducibility.

In [ ]:
fit_indices, val_indices = train_test_split(
    np.arange(len(train_df)), test_size=0.1, random_state=SEED,
    stratify=train_df['category'])
fitting_df = train_df.iloc[fit_indices].copy()
validation_df = train_df.iloc[val_indices].copy()
fitting_df.to_csv(OUTPUT_DIR / 'xlmr_fitting_data.csv', index=False, encoding='utf-8-sig')
validation_df.to_csv(OUTPUT_DIR / 'xlmr_validation_data.csv', index=False, encoding='utf-8-sig')
print(f'Fitting: {len(fitting_df)}; validation: {len(validation_df)}; test: {len(test_df)}')


## Step 6 — Load tokenizer and model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)


## Step 7 — Tokenize fitting and validation data
The original 256-token truncation and fixed padding are retained. Long articles are truncated; this is not a 256-word limit.

In [ ]:
def tokenize_frame(frame):
    dataset = Dataset.from_dict({
        'text': frame['text'].tolist(),
        'labels': [label2id[label] for label in frame['category']]})
    def preprocess(examples):
        return tokenizer(examples['text'], truncation=True,
                         max_length=MAX_LENGTH, padding='max_length')
    return dataset.map(preprocess, batched=True, remove_columns=['text'])

tokenized_train = tokenize_frame(fitting_df)
tokenized_validation = tokenize_frame(validation_df)


## Step 8 — Define metrics
Macro-F1 weights all categories equally. These metrics are calculated on validation data during training; test metrics are calculated only after checkpoint selection.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    class_ids = list(range(len(label_list)))
    return {
        'accuracy': accuracy_score(labels, predictions),
        'macro_f1': f1_score(labels, predictions, labels=class_ids, average='macro', zero_division=0),
        'micro_f1': f1_score(labels, predictions, labels=class_ids, average='micro', zero_division=0),
        'weighted_f1': f1_score(labels, predictions, labels=class_ids, average='weighted', zero_division=0),
        'macro_precision': precision_score(labels, predictions, labels=class_ids, average='macro', zero_division=0),
        'macro_recall': recall_score(labels, predictions, labels=class_ids, average='macro', zero_division=0)}


## Step 9 — Configure training
Four epochs, training batch size 16, evaluation batch size 32, accumulation 2, warmup 500, and weight decay 0.01 are retained. learning rate is made explicit as 5e-5. FP16 is enabled only on CUDA. At most two checkpoints are retained to limit disk use. Training runs for all four epochs; there is no early-stopping callback.

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / 'checkpoints'),
    num_train_epochs=4,
    learning_rate=5e-5,
    lr_scheduler_type='linear',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    data_seed=SEED,
    report_to='none')

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,  # IMPORTANT: validation, not test
    processing_class=tokenizer,
    compute_metrics=compute_metrics)


## Step 10 — Train and save selected checkpoint
The best model is chosen by validation macro-F1. Model files and tokenizer are saved to Drive. The test set is not involved in selecting an epoch.

In [ ]:
start = time.perf_counter()
trainer.train()
training_seconds = time.perf_counter() - start
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best validation macro-F1:', trainer.state.best_metric)
trainer.save_model(str(OUTPUT_DIR / 'best_model'))
tokenizer.save_pretrained(str(OUTPUT_DIR / 'best_model'))
trainer.state.save_to_json(str(OUTPUT_DIR / 'trainer_state.json'))


## Step 11 — Evaluate the shared test set
This evaluates the selected model. Do not choose epochs or change settings based on these test results.

In [ ]:
tokenized_test = tokenize_frame(test_df)
start = time.perf_counter()
prediction_output = trainer.predict(tokenized_test, metric_key_prefix='test')
test_prediction_seconds = time.perf_counter() - start
logits = prediction_output.predictions
if isinstance(logits, tuple):
    logits = logits[0]
pred_ids = np.argmax(logits, axis=-1)
pred_names = [id2label[int(i)] for i in pred_ids]
results = {key: float(value) for key, value in prediction_output.metrics.items()}
print(json.dumps(results, indent=2))
report = classification_report(test_df['category'], pred_names, labels=label_list,
                               digits=4, zero_division=0)
report_json = classification_report(test_df['category'], pred_names, labels=label_list,
                                    output_dict=True, zero_division=0)
print(report)


## Step 12 — Save reports, predictions, and configuration
Predictions retain article IDs so they can be matched with LR, SVM, MLP, and FastText. Training time includes epoch validation and checkpoint saving; it is not directly equivalent to the baseline fit-only timing.

In [ ]:
(OUTPUT_DIR / 'XLMR_report.txt').write_text(report, encoding='utf-8')
(OUTPUT_DIR / 'XLMR_report.json').write_text(json.dumps(report_json, indent=2), encoding='utf-8')
(OUTPUT_DIR / 'test_metrics.json').write_text(json.dumps(results, indent=2), encoding='utf-8')
prediction_table = test_df[['article_id', 'category']].rename(columns={'category': 'true_label'}).copy()
prediction_table['XLMR'] = pred_names
prediction_table.to_csv(OUTPUT_DIR / 'test_predictions.csv', index=False, encoding='utf-8-sig')
cm = confusion_matrix(test_df['category'], pred_names, labels=label_list)
pd.DataFrame(cm, index=label_list, columns=label_list).to_csv(OUTPUT_DIR / 'XLMR_confusion.csv')
pd.DataFrame([dict(model='XLMR', accuracy=accuracy_score(test_df['category'], pred_names),
                   macro_precision=report_json['macro avg']['precision'],
                   macro_recall=report_json['macro avg']['recall'],
                   macro_f1=report_json['macro avg']['f1-score'],
                   weighted_f1=report_json['weighted avg']['f1-score'],
                   training_seconds=training_seconds)]).to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)
metadata = dict(
    model=MODEL_NAME, model_revision=getattr(model.config, '_commit_hash', None),
    seed=SEED, max_length=MAX_LENGTH, validation_fraction_of_shared_train=0.1,
    shared_train_records=len(train_df), fitting_records=len(fitting_df),
    validation_records=len(validation_df), test_records=len(test_df),
    label2id=label2id, best_checkpoint=trainer.state.best_model_checkpoint,
    best_validation_macro_f1=trainer.state.best_metric,
    train_file_sha256=hashlib.sha256(TRAIN_PATH.read_bytes()).hexdigest(),
    test_file_sha256=hashlib.sha256(TEST_PATH.read_bytes()).hexdigest(),
    training_arguments=training_args.to_dict(),
    versions=dict(python=platform.python_version(), torch=torch.__version__,
                  transformers=transformers.__version__, datasets=datasets.__version__,
                  sklearn=sklearn.__version__, pandas=pd.__version__),
    training_seconds=training_seconds, test_prediction_seconds=test_prediction_seconds,
    timing_note='Training includes validation and checkpoint writes; tokenization is excluded.')
(OUTPUT_DIR / 'experiment_config.json').write_text(json.dumps(metadata, indent=2, default=str), encoding='utf-8')
print('Saved results and selected model to:', OUTPUT_DIR)


## Step 13 — Plot the confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_list, yticklabels=label_list)
plt.title('XLM-R: shared test set')
plt.ylabel('True category')
plt.xlabel('Predicted category')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_xlmr.png', dpi=300)
plt.show()


## Step 14 — Download reports
The selected model and all outputs already remain in Drive. This zip contains reports and predictions only, excluding the large model and checkpoints.

In [ ]:
import zipfile
from google.colab import files
zip_path = Path('/content/xlmr_reports.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in ('XLMR_report.txt', 'XLMR_report.json', 'test_metrics.json',
                 'test_predictions.csv', 'XLMR_confusion.csv', 'model_comparison.csv',
                 'experiment_config.json', 'trainer_state.json', 'confusion_matrix_xlmr.png'):
        archive.write(OUTPUT_DIR / name, arcname=name)
files.download(str(zip_path))
